# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² protein abundance dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access basic metadata (without subscripting metadata)
print(f"Dataset Title: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Published Date: {dataset.metadata.date_published}")
print(f"Dataset License: {dataset.metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We use the Croissant metadata to enumerate the dataset's top-level components following the `@id` convention.

In [ ]:
# List record sets and their properties
print('Available Record Sets:')
record_sets = []
for rs in dataset.record_sets:
    print(f"  RecordSet @id: {rs.id}")
    print(f"    Name: {rs.name if hasattr(rs, 'name') else '(no name)'}")
    print(f"    Description: {rs.description if hasattr(rs, 'description') else '(no description)'}")
    print(f"    Fields (@id):")
    for field in rs.fields:
        print(f"      - {field.id} (name: {field.name})")
    record_sets.append(rs.id)

# For reference, print all field @ids from the first record set
if record_sets:
    first_rs_fields = dataset.get_record_set(record_sets[0]).fields
    print(f'Fields in first record set ({record_sets[0]}):')
    for field in first_rs_fields:
        print(f"  Field @id: {field.id}, name: {field.name}, data_type: {getattr(field, 'data_type', 'unknown')}")

## 3. Data Extraction
Load data from the main record set (by its `@id`) into a DataFrame for analysis. Keys and field `@id`s are used for reference.

_Note: You may need to adjust the `record_set_id` variable depending on which record set you want to explore._

In [ ]:
# Use the main record set for this dataset.
main_record_set_id = record_sets[0] if record_sets else None

# Extract all records as a DataFrame
df = None
if main_record_set_id:
    records = list(dataset.records(record_set=main_record_set_id))
    df = pd.DataFrame(records)
    print(f'Columns in record set {main_record_set_id}:')
    print(df.columns.tolist())
    display(df.head())
else:
    print('No record set found to extract!')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records on a numeric field, normalization, and grouping by a categorical field.

_We use only field `@id`s to ensure traceability to the schema._

In [ ]:
# Select fields by their @id for further processing
if main_record_set_id and df is not None:
    # Print all available field @ids for inspection
    print('All column/field names (should be Croissant @id):')
    print(df.columns.tolist())

    # Example field IDs (replace with actual @ids from above printout if different)
    # Let's try to pick a numeric column and a group/categorical column.
    # Suppose 'cr:field:normalized_abundance' and 'cr:field:protein_id' are present as example @ids

    # Update below these variables once actual field @ids are seen from section above.
    numeric_field_id = None
    group_field_id = None
    fallback_numeric_fields = [c for c in df.columns if ('abundance' in c or 'count' in c or 'coverage' in c or 'mass' in c or 'mw' in c) and pd.api.types.is_numeric_dtype(df[c])]
    fallback_group_fields = [c for c in df.columns if any(k in c.lower() for k in ['group', 'sample', 'type', 'protein'])]
    if fallback_numeric_fields:
        numeric_field_id = fallback_numeric_fields[0]
    if fallback_group_fields:
        group_field_id = fallback_group_fields[0]
    print(f"Selected numeric field: {numeric_field_id}")
    print(f"Selected group/categorical field: {group_field_id}")

    # Filter out records with low value in the numeric field
    if numeric_field_id and pd.api.types.is_numeric_dtype(df[numeric_field_id]):
        threshold = df[numeric_field_id].mean() # as example, above mean
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Records with {numeric_field_id} > {threshold:.4f} (mean): {len(filtered_df)}")

        # Normalize the numeric field
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"First few normalized {numeric_field_id} values:")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Group by a categorical/ID field and take the mean of the normalized value
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[norm_col].mean().reset_index()
            print(f"Grouped (mean-normalized) by {group_field_id}:")
            display(grouped_df.head())
    else:
        print('No suitable numeric field found for analysis.')
else:
    print('No suitable data to analyze!')

## 5. Visualization
Visualize one or more distributions from the data.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

if main_record_set_id and df is not None and numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    # If group/categorical field is present, boxplot by group
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10, 5))
        # Only plot if few unique groups
        if df[group_field_id].nunique() < 25:
            sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
            plt.title(f"{numeric_field_id} by {group_field_id}")
            plt.xticks(rotation=45)
            plt.show()
else:
    print('No numeric field selected for visualization.')

## 6. Conclusion
In this notebook, we have loaded and explored the FAIR² protein dataset via its Croissant schema. We listed its record sets and fields, extracted the main table into a DataFrame, processed and filtered a numeric field by its `@id`, normalized and grouped the data, and visualized distributions. For deeper analyses, consult the Croissant schema to map all field `@id` values explicitly.

<sup>Developed with the [`mlcroissant`](https://github.com/mlcommons/croissant) library.</sup>